In [ ]:
from tensorflow import keras # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from datetime import datetime
import os
import pandas as pd
import numpy as np

In [ ]:
#Opens a new dataframe with the Clean csv
cleancsv = pd.read_csv('CSV/CLEAN.csv')

In [ ]:
#Convert data into Date time and create date filter
cleancsv['Date'] = pd.to_datetime(cleancsv['Date'])
cleancsv['Date'] = cleancsv['Date'] + pd.to_timedelta(cleancsv["Hr"], unit="h")
cleancsv.drop('Hr', axis=1, inplace=True)

"""
Use this in future if data set needs specific dates
prediction = data.loc{
    (untouched_csv['Date'] > datetime(x, x, x)) &
    (untouched_csv['Date'] < datetime(x, x, x,))
}
"""

"\nUse this in future if data set needs specific dates\nprediction = data.loc{\n    (untouched_csv['Date'] > datetime(x, x, x)) &\n    (untouched_csv['Date'] < datetime(x, x, x,))\n}\n"

In [ ]:
#Create month colomn and restrict to only summer months
summer_mask = cleancsv["Date"].dt.month.isin([6, 7, 8])
cleancsv = cleancsv[summer_mask].reset_index(drop=True)

In [ ]:
#Prepare colomns into variables
data_main_air_temp = cleancsv['Mainland Air Temp']
data_humidity_per = cleancsv['Humidity (%)']
data_wind_direction = cleancsv['Direction (A)']
data_wind_speed = cleancsv['Wind Speed (A)']
data_gusting = cleancsv['Gusting']
data_pressure = cleancsv['Atmospheric Pressure (IN)']
data_rainfall = cleancsv['Precipitation Rate']
data_bay_temp = cleancsv['Bay Temp']
data_salinity = cleancsv['Salinity']
data_lbi_temp = cleancsv['LBI Air Temp']
data_ocean_temp = cleancsv['Ocean Temp']
data_onshore_flag = cleancsv['Onshore']
data_upwelling_flag = cleancsv['upwelling_flag']

#saves all input data into one Numpy array
dataset = np.column_stack([
    data_main_air_temp.values,
    data_humidity_per.values,
    data_wind_direction.values,
    #data_wind_speed.values,
    data_gusting.values,
    data_pressure.values,
    data_rainfall.values,
    data_bay_temp.values,
    data_salinity.values,
    data_lbi_temp.values,
    data_ocean_temp.values,
    data_onshore_flag.values,
    data_upwelling_flag.values,
])

#Save output data into variables and reshape it to be a 2d array
output_data = data_wind_speed.values
output_data = np.array(output_data).reshape(-1, 1)

In [ ]:
#Length of training data
training_data_len = int(np.ceil(len(dataset) * 0.90)) #Use 90% of training data

In [ ]:
#Scaler
scaler_x= StandardScaler()
scaler_y= StandardScaler()


scaledx = scaler_x.fit_transform(dataset)
scaledy = scaler_y.fit_transform(output_data)

# X: last time step features only
X_all = scaledx          # (N, 12)
y_all = output_data


In [ ]:
X_train = X_all[:training_data_len]
y_train = y_all[:training_data_len]
X_test  = X_all[training_data_len:]
y_test  = y_all[training_data_len:]

In [ ]:
#Build the model
model = keras.models.Sequential()

In [ ]:
#Layer Zero input_shape=(X_train.shape[1], 1)
model.add(keras.layers.Input(shape=(X_train.shape[1])))

In [ ]:
#First Layer input_shape=(X_train.shape[1], 1)
#model.add(keras.layers.LSTM(64, return_sequences=True))

In [ ]:
#Second Layer
model.add(keras.layers.Dense(32, activation="relu"))

In [ ]:
#3rd Layer (Dense)
model.add(keras.layers.Dense(16, activation="relu"))

In [ ]:
#4th Layer (Dropout)
#model.add(keras.layers.Dropout(0.5))

In [ ]:
#Final Output Layer (Dense)
model.add(keras.layers.Dense(1))

In [ ]:
#Put all the layers together
model.summary()
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss="mse",
              metrics=[keras.metrics.RootMeanSquaredError()])

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_16 (LSTM)                  │ (None, 32)             │         5,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,849 (26.75 KB)

 Trainable params: 6,849 (26.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Train the model

#epochs = # of runs
#batch size = how much data is in each batch
training = model.fit(
X_train,
    y_train,
    epochs=80,
    batch_size=32,
    validation_split=0.1,
    shuffle=True,
    verbose=0
)

Epoch 1/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 4.2389 - root_mean_squared_error: 5.0581 - val_loss: 4.4480 - val_root_mean_squared_error: 5.1859
Epoch 2/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.1736 - root_mean_squared_error: 5.0022 - val_loss: 4.3917 - val_root_mean_squared_error: 5.1363
Epoch 3/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.1172 - root_mean_squared_error: 4.9545 - val_loss: 4.3360 - val_root_mean_squared_error: 5.0873
Epoch 4/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.0611 - root_mean_squared_error: 4.9063 - val_loss: 4.2801 - val_root_mean_squared_error: 5.0384
Epoch 5/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.0055 - root_mean_squared_error: 4.8590 - val_loss: 4.2244 - val_root_mean_squared_error: 4.9898
Epoch 6/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.9499 - root_mean_squared_error: 4.8117 - val_loss: 4.1688 - val_root_mean_squared_error: 4.9416
Epoch 7/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.8947

In [ ]:
pred = model.predict(X_test)
print("Pred mean/std:", pred.mean(), pred.std())
print("Test MAE:", mean_absolute_error(y_test.ravel(), pred.ravel()))